In [2]:
import gensim
import gensim.corpora as corpora
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from gensim.models.ldamodel import LdaModel

from pprint import pprint

import spacy
from tqdm.notebook import tqdm
import sys

import pickle
import re
import pyLDAvis
import pyLDAvis.gensim
import string
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import seaborn as sns

import wordcloud
from wordcloud import WordCloud
import pyLDAvis.gensim_models as gensimvis
from tqdm import tqdm
from collections import Counter
from pathlib import Path

nlp = spacy.load('en_core_web_sm')

In [3]:
PROCESSED_FACTS_PATH = Path("..") / "dataset" / "processed_facts.csv"
RESULTS_PATH = Path("..") / "results" / "lda_df_03.csv"
VIS_PATH = Path("..") / "figures" 

Load the data and turn it back into a list

In [ ]:
clean_facts_bigrams = pd.read_csv(PROCESSED_FACTS_PATH, index_col = 0)
clean_facts_bigrams = [row.split() for row in clean_facts_bigrams['facts']]
clean_facts_bigrams


Metric calculation functions

In [48]:
def compute_coherence_values_lda(facts, corpus, dictionary, k, a, b):
    """
    Trains an LDA model and computes its coherence score.

    Args:
        facts (list): Preprocessed text data.
        corpus (list): Corpus created from text data.
        dictionary (gensim.corpora.Dictionary): Gensim dictionary.
        k (int): Number of topics.
        a (float): Alpha parameter.
        b (float): Beta parameter.

    Returns:
        float: Coherence score.
    """
    lda_model = LdaModel(corpus=corpus, num_topics=k, id2word=dictionary, 
                         passes=10, alpha=a, eta=b, random_state=42)
    coherence_model = CoherenceModel(model=lda_model, texts=facts, 
                                     dictionary=dictionary, coherence='c_v')
    return coherence_model.get_coherence()


In [49]:
def calculate_topic_diversity(model, top_n: int) -> float:
    """
    Calculates topic diversity score based on the number of unique words across topics.

    Args:
        model (gensim.models.LdaModel): Trained LDA model.
        top_n (int): Number of top words to consider per topic.

    Returns:
        float: Topic diversity score.
    """
    topic_words = [word for topic_id in range(model.num_topics) 
                   for word, _ in model.show_topic(topic_id, topn=top_n)]
    unique_words = set(topic_words)
    return len(unique_words) / (model.num_topics * top_n)

In [51]:
def calculate_coherence(model, facts: list, dictionary, metric: str) -> float:
    """
    Calculates the coherence score for a given LDA model.

    Args:
        model (gensim.models.LdaModel): The trained LDA model for which coherence is calculated.
        facts (list): List of preprocessed text data, where each element is a tokenized document.
        dictionary (gensim.corpora.Dictionary): Gensim dictionary object used to create the corpus.
        metric (str): The coherence metric to use. Common options include:
                      - 'u_mass': Based on word co-occurrence in a sliding window.
                      - 'c_v': Based on a sliding window and normalized pointwise mutual information (NPMI).
                      - 'c_uci': Based on pointwise mutual information (PMI).
                      - 'c_npmi': Normalized PMI.

    Returns:
        float: The coherence score for the given model using the specified metric.
    """
    coherence_model = CoherenceModel(model=model, texts=facts,
                                     dictionary=dictionary, coherence=metric)
    return coherence_model.get_coherence()


In [52]:
def calculate_topic_diversity_for_row(row, corpus: list, dictionary) -> float:
    """
    Calculates the topic diversity score for a specific row of hyperparameters 
    in a parameter grid search.

    Args:
        row (pd.Series): A row from a parameter grid search DataFrame containing:
                         - 'Topics' (int): Number of topics.
                         - 'Alpha' (float or str): Alpha hyperparameter value (e.g., 'symmetric').
                         - 'Beta' (float or str): Beta hyperparameter value (e.g., 'symmetric').
        corpus (list): The bag-of-words representation of the text data.
        dictionary (gensim.corpora.Dictionary): Gensim dictionary object used to create the corpus.

    Returns:
        float: The topic diversity score for the LDA model trained with the given hyperparameters.
    """
    k = int(row['Topics'])
    alpha = row['Alpha']
    beta = row['Beta']

    lda_model = LdaModel(corpus=corpus, num_topics=k,
                         id2word=dictionary, passes=10,
                         alpha=alpha, eta=beta, chunksize=100,
                         per_word_topics=True, random_state=42)
    
    return calculate_topic_diversity(lda_model, top_n=10)


In [54]:
def remove_frequent(facts, max_df: float):
    """
    Removes frequent words that appear in more than a specified percentage of documents.

    Args:
        facts (list of list of str): Preprocessed text data, where each document is a list of tokens.
        max_df (float): Maximum percentage (between 0 and 1) of documents in which a word can appear 
                        before being filtered out. For example, 0.3 means remove words appearing in more 
                        than 30% of documents.

    Returns:
        tuple:
            - dictionary (gensim.corpora.Dictionary): A dictionary mapping tokens to IDs, excluding frequent words.
            - corpus (list of list of tuple): Bag-of-words representation of the filtered text data.
    """
    dictionary = corpora.Dictionary(facts)
    dictionary.filter_extremes(no_above=max_df)
    corpus = [dictionary.doc2bow(text) for text in facts]
    
    return dictionary, corpus


In [56]:
def run_lda_tests(facts, dictionary, corpus, min_topics: int, max_topics: int, step_size: int = 1):
    """
    Runs grid search over topic, alpha, and beta parameters for LDA, evaluating coherence.

    Args:
        facts (list): Preprocessed text data.
        dictionary (gensim.corpora.Dictionary): Gensim dictionary.
        corpus (list): Corpus created from text data.
        min_topics (int): Minimum number of topics.
        max_topics (int): Maximum number of topics.
        step_size (int): Increment for the number of topics.

    Returns:
        pd.DataFrame: Results of grid search with coherence scores.
    """
    topics_range = range(min_topics, max_topics + 1, step_size)
    alpha_values = list(np.arange(0.01, 1, 0.3)) + ['symmetric', 'asymmetric']
    beta_values = list(np.arange(0.01, 1, 0.3)) + ['symmetric']

    results = {'Topics': [], 'Alpha': [], 'Beta': [], 'Coherence': []}
    total_iterations = len(topics_range) * len(alpha_values) * len(beta_values)

    with tqdm(total=total_iterations) as pbar:
        for k in topics_range:
            for alpha in alpha_values:
                for beta in beta_values:
                    coherence = compute_coherence_values_lda(facts, corpus, dictionary, k, alpha, beta)
                    results['Topics'].append(k)
                    results['Alpha'].append(alpha)
                    results['Beta'].append(beta)
                    results['Coherence'].append(coherence)
                    pbar.update(1)

    return pd.DataFrame(results)

Run the experiments for max_df value of 0.3

In [57]:
dictionary_0_3, corpus_0_3 = remove_frequent(clean_facts_bigrams,0.3)
df_0_3 = run_lda_tests(clean_facts_bigrams, dictionary_0_3, corpus_0_3, 1, 20)


100%|██████████| 600/600 [4:47:40<00:00, 28.77s/it]   


Save the results in a csv file 

In [59]:
def analyze_and_save_results(results: pd.DataFrame, corpus, dictionary, filename: str) -> pd.DataFrame:
    """
    Analyzes and saves the results of the LDA tests.

    Args:
        results (pd.DataFrame): DataFrame containing LDA test results with topics, alpha, beta, and coherence.
        corpus (list): Bag-of-words representation of the text data.
        dictionary (gensim.corpora.Dictionary): Gensim dictionary object used to create the corpus.
        filename (str): Name of the CSV file to save results.

    Returns:
        pd.DataFrame: The DataFrame sorted by Topic Quality (descending).
    """
    results['Topic Diversity'] = results.apply(
        lambda row: calculate_topic_diversity_for_row(row, corpus, dictionary), axis=1
    )
    results['Topic Quality'] = results['Topic Diversity'] * results['Coherence']
    results = results.sort_values(by='Topic Quality', ascending=False)
    results.to_csv(filename, index=False)

    return results


In [61]:
analyze_and_save_results(df_0_3, corpus_0_3, dictionary_0_3, RESULTS_PATH)

,Topics,Alpha,Beta,Coherence,Topic Diversity,Topic Quality
163,6,0.61,0.91,0.400271,0.966667,0.386929
205,7,asymmetric,0.01,0.390798,0.985714,0.385215
158,6,0.31,0.91,0.396656,0.966667,0.383434
157,6,0.31,0.61,0.396415,0.966667,0.383201
254,9,0.61,symmetric,0.395286,0.966667,0.382110
...,...,...,...,...,...,...
15,1,0.91,0.01,0.198247,1.000000,0.198247
14,1,0.61,symmetric,0.198247,1.000000,0.198247
13,1,0.61,0.91,0.198247,1.000000,0.198247
12,1,0.61,0.61,0.198247,1.000000,0.198247


In [ ]:
def load_results(filename: str) -> pd.DataFrame:
    """
    Loads LDA test results from a CSV file for further analysis.

    Args:
        filename (str): Path to the CSV file containing results.

    Returns:
        pd.DataFrame: The loaded results DataFrame.
    """
    results = pd.read_csv(filename)
    results = results.sort_values(by='Topic Quality', ascending=False)
    return results


Show words per topic and pyLDAvisualisation for the two best performing models

In [63]:
best_model_performing = LdaModel(corpus=corpus_0_3, num_topics=6,
                                id2word=dictionary_0_3, passes=10,
                                alpha=0.61, eta=0.91, chunksize=100,
                                per_word_topics=True, random_state=42)

pprint(best_model_performing.print_topics(num_words=30))


[(0,
  '0.008*"programme" + 0.007*"broadcast" + 0.005*"advertising" + '
  '0.005*"policy" + 0.005*"vote" + 0.004*"system" + 0.004*"election" + '
  '0.004*"candidate" + 0.004*"parliamentary" + 0.004*"advertisement" + '
  '0.004*"broadcasting" + 0.004*"appoint" + 0.003*"appointment" + '
  '0.003*"prohibition" + 0.003*"organisation" + 0.003*"restriction" + '
  '0.003*"legislation" + 0.003*"fundamental" + 0.003*"ensure" + '
  '0.003*"television" + 0.003*"trade" + 0.003*"ban" + 0.003*"employee" + '
  '0.003*"management" + 0.003*"free" + 0.003*"status" + 0.003*"change" + '
  '0.003*"electoral" + 0.003*"debate" + 0.003*"amend"'),
 (1,
  '0.006*"secret" + 0.005*"medical" + 0.005*"cassation" + 0.005*"trial" + '
  '0.004*"minister" + 0.004*"disciplinary" + 0.004*"document" + '
  '0.004*"prisoner" + 0.004*"attorney_general" + 0.004*"code" + '
  '0.003*"patient" + 0.003*"lawyer" + 0.003*"treatment" + 0.003*"release" + '
  '0.003*"doctor" + 0.003*"principle" + 0.003*"prosecutor" + '
  '0.003*"injun

In [64]:
pyLDAvis.enable_notebook()
p = gensimvis.prepare(best_model_performing, corpus_0_3, dictionary_0_3, mds='mmds')
p


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2      0.039942  0.155571       1        1  23.709725
0      0.034205 -0.138445       2        1  18.132608
5     -0.154610  0.097345       3        1  17.267571
1      0.025508  0.019995       4        1  16.117054
4      0.175140 -0.025708       5        1  13.657761
3     -0.120186 -0.108758       6        1  11.115281, topic_info=               Term        Freq       Total Category  logprob  loglift
2335   private_life  916.000000  916.000000  Default  30.0000  30.0000
1530      religious  810.000000  810.000000  Default  29.0000  29.0000
2034      programme  869.000000  869.000000  Default  28.0000  28.0000
1877          datum  514.000000  514.000000  Default  27.0000  27.0000
669          source  810.000000  810.000000  Default  26.0000  26.0000
...             ...         ...         ...      ...      ...      ...
707   dissemination  146.153981  469.714782   Topic6  -5.9435   1.0294
3245      publisher  135.063248  376.144349   Topic6  -6.0224   1.1726
321        identify  139.009758  442.789605   Topic6  -5.9936   1.0383
914     restriction  153.107691  871.995349   Topic6  -5.8970   0.4572
488          system  142.720110  770.256246   Topic6  -5.9673   0.5110

[412 rows x 6 columns], token_table=      Topic      Freq      Term
term                           
3023      1  0.016454  abortion
3023      2  0.016454  abortion
3023      3  0.016454  abortion
3023      4  0.921425  abortion
3023      5  0.016454  abortion
...     ...       ...       ...
2906      2  0.034992     world
2906      3  0.009998     world
2906      4  0.124972     world
2906      5  0.634860     world
2906      6  0.054988     world

[2130 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[3, 1, 6, 2, 5, 4])

In [ ]:
save_path = VIS_PATH / 'lda_topic_number_6.html'
pyLDAvis.save_html(p, save_path )

In [67]:
second_best_model_performing = LdaModel(corpus=corpus_0_3, num_topics=7,
                                id2word=dictionary_0_3, passes=10,
                                alpha='asymmetric', eta=0.01, chunksize=100,
                                per_word_topics=True, random_state=42)

pprint(second_best_model_performing.print_topics(num_words=30))


[(0,
  '0.010*"programme" + 0.008*"broadcast" + 0.007*"appoint" + 0.006*"candidate" '
  '+ 0.006*"vote" + 0.006*"parliamentary" + 0.005*"election" + 0.005*"system" '
  '+ 0.005*"code" + 0.005*"legislation" + 0.004*"organisation" + '
  '0.004*"broadcasting" + 0.004*"amend" + 0.004*"appointment" + 0.004*"status" '
  '+ 0.004*"minister" + 0.004*"fundamental" + 0.004*"change" + '
  '0.004*"disciplinary" + 0.004*"ensure" + 0.004*"restriction" + '
  '0.004*"employee" + 0.004*"management" + 0.003*"amendment" + 0.003*"elect" + '
  '0.003*"free" + 0.003*"propose" + 0.003*"requirement" + 0.003*"meeting" + '
  '0.003*"representative"'),
 (1,
  '0.009*"secret" + 0.005*"military" + 0.005*"principle" + 0.005*"document" + '
  '0.005*"attorney_general" + 0.005*"prisoner" + 0.004*"servant" + '
  '0.004*"deal" + 0.004*"special" + 0.004*"trial" + 0.003*"security" + '
  '0.003*"reasonable" + 0.003*"secrecy" + 0.003*"responsibility" + '
  '0.003*"clear" + 0.003*"disclosure" + 0.003*"extend" + 0.003*"contra

In [68]:
pyLDAvis.enable_notebook()
p = gensimvis.prepare(second_best_model_performing, corpus_0_3, dictionary_0_3, mds='mmds')
p


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2      0.066218 -0.196319       1        1  22.490058
0      0.027010  0.260395       2        1  17.886701
6     -0.155266 -0.249127       3        1  14.423323
1     -0.102267  0.141242       4        1  14.354059
4      0.227426  0.153770       5        1  12.732162
3      0.291988 -0.086034       6        1   9.463801
5     -0.355109 -0.023927       7        1   8.649897, topic_info=              Term         Freq        Total Category  logprob  loglift
641     prosecutor  2067.000000  2067.000000  Default  30.0000  30.0000
2335  private_life  1167.000000  1167.000000  Default  29.0000  29.0000
1530     religious   959.000000   959.000000  Default  28.0000  28.0000
669         source  1013.000000  1013.000000  Default  27.0000  27.0000
1502       meeting  1003.000000  1003.000000  Default  26.0000  26.0000
...            ...          ...          ...      ...      ...      ...
1903      organise   179.953140   488.666743   Topic7  -5.4847   1.4486
579         doctor   176.743790   577.688030   Topic7  -5.5027   1.2633
108        protest   165.825431   371.412348   Topic7  -5.5665   1.6412
557       building   162.757069   396.928442   Topic7  -5.5851   1.5561
1841         start   160.565855   542.415207   Topic7  -5.5987   1.2303

[519 rows x 6 columns], token_table=      Topic      Freq        Term
term                             
3023      4  0.999272    abortion
2357      3  1.000289     abscond
1382      2  0.012853      access
1382      6  0.989695      access
1712      4  0.083874  accessible
...     ...       ...         ...
2906      1  0.142281       world
2906      4  0.073296       world
2906      5  0.741588       world
2906      6  0.047427       world
3531      3  0.989160  write_oral

[1146 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[3, 1, 7, 2, 5, 4, 6])

In [ ]:
save_path = VIS_PATH / 'lda_topic_number_7.html'
pyLDAvis.save_html(p, save_path )
